# Geração de documentação Markdown para RAG — HighBond API

Este notebook analisa o arquivo `openapi.json` (HighBond API Reference) e gera documentação em Markdown com TODA a informação de cada endpoint, pronta para uso em um sistema de RAG.

**O que é gerado:**
1. **Chunks por domínio** — um `.md` por tag/domínio em `raw_info/chunks/`, cada um com a descrição do recurso + referência completa de todos os endpoints do domínio. Inclui `index.md`.
2. **Documento único** — `raw_info/highbond_api_completo.md` com índice + conceitos + todos os endpoints.

**Para cada endpoint**, ambos os formatos incluem: método, path, `operationId`, resumo, descrição completa, e seções **colapsáveis** (`<details>`, todas fechadas por padrão) com:
- **Parâmetros** (path/query) com tipo, obrigatoriedade, enum, default e exemplo — incluindo `$ref` resolvidos;
- **Request body (payload)** com o schema completo em tabela (campos, tipos, obrigatórios, enums, exemplos), com `$ref`/`anyOf`/`oneOf` resolvidos recursivamente;
- **Respostas** por status code, cada uma colapsável, com schema de resposta quando documentado.

Sem classificação de importância — a relevância será definida pelos embeddings no RAG.

In [25]:
import json
import re
from pathlib import Path

SPEC_PATH = Path("openapi.json")
CHUNKS_DIR = Path("raw_info") / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = Path("raw_info") / "highbond_api_completo.md"

spec = json.loads(SPEC_PATH.read_text(encoding="utf-8"))

total_ops = sum(
    1
    for methods in spec["paths"].values()
    for op in methods.values()
    if isinstance(op, dict) and "operationId" in op
)
print(f"Spec carregada: {spec['info']['title']} v{spec['info']['version']}")
print(f"Endpoints: {total_ops} | Tags: {len(spec['tags'])} | Schemas: {len(spec['components']['schemas'])}")

Spec carregada: HighBond API Reference v1.1.0
Endpoints: 377 | Tags: 62 | Schemas: 472


## Funções auxiliares
Limpeza de HTML (as descrições do spec contêm tags), resolução de `$ref` e renderização de schemas/parâmetros/payloads/respostas em tabelas Markdown.

In [26]:
def clean_html(text: str) -> str:
    """Remove tags HTML, preservando o texto de links e blocos de código."""
    if not text:
        return ""
    # Links: <a href="..." target="_blank">texto</a> -> texto
    text = re.sub(r'<a\s+[^>]*>(.*?)</a>', r'\1', text, flags=re.S)
    # Demais tags (strong, ul/li viram bullets, br vira quebra de linha)
    text = re.sub(r'<br\s*/?>', '\n', text)
    text = re.sub(r'<li>', '\n- ', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def slugify(text: str) -> str:
    slug = re.sub(r'[^a-z0-9]+', '-', text.lower()).strip('-')
    return slug[:80]


def split_h2_sections(md_text: str) -> list[tuple[str, str]]:
    """Divide um markdown em (titulo, corpo) por headings nível 2 (## )."""
    parts = re.split(r'(?m)^##\s+', md_text)
    sections = []
    for part in parts[1:]:  # ignora conteúdo antes do primeiro ##
        title, _, body = part.partition('\n')
        sections.append((title.strip(), body.strip()))
    return sections


def iter_operations():
    """Itera (path, method, operation) sobre todos os endpoints do spec."""
    for path, methods in spec["paths"].items():
        for method, op in methods.items():
            if isinstance(op, dict) and "operationId" in op:
                yield path, method.upper(), op


# ---------------------------------------------------------------------------
# Resolução de $ref e renderização de schemas
# ---------------------------------------------------------------------------
MAX_DEPTH = 6  # profundidade máxima de aninhamento renderizado


def resolve_ref(ref: str) -> tuple[str, dict]:
    """Resolve um $ref local ('#/components/...'). Retorna (nome, objeto)."""
    parts = ref.lstrip("#/").split("/")
    node = spec
    for p in parts:
        node = node[p]
    return parts[-1], node


def schema_type_str(schema: dict) -> str:
    """Representação curta do tipo de um schema."""
    if not isinstance(schema, dict):
        return ""
    if "$ref" in schema:
        return resolve_ref(schema["$ref"])[0]
    for key in ("anyOf", "oneOf"):
        if key in schema:
            variants = " | ".join(schema_type_str(s) for s in schema[key])
            return f"{key}: {variants}"
    t = schema.get("type", "")
    if t == "array":
        return f"array[{schema_type_str(schema.get('items', {}))}]"
    return t or "object"


def schema_rows(schema, prefix="", ancestors=frozenset(), depth=0):
    """Achata as propriedades de um schema em linhas (path, tipo, obrigatório, descrição)."""
    rows = []
    if not isinstance(schema, dict):
        return rows
    if "$ref" in schema:
        ref_name, target = resolve_ref(schema["$ref"])
        if ref_name in ancestors or depth >= MAX_DEPTH:
            return [(prefix.rstrip("."), ref_name, "", "↩ *referência recursiva — ver schema* `" + ref_name + "`")]
        return schema_rows(target, prefix, ancestors | {ref_name}, depth + 1)
    for key in ("anyOf", "oneOf"):
        if key in schema:
            for variant in schema[key]:
                rows += schema_rows(variant, prefix, ancestors, depth)
            return rows
    required = set(schema.get("required", []))
    for name, prop in (schema.get("properties") or {}).items():
        if not isinstance(prop, dict):
            prop = {}
        ptype = schema_type_str(prop)
        desc = clean_html(prop.get("description", "")).replace("\n", " ").replace("|", "\\|")
        if prop.get("enum"):
            desc += (" " if desc else "") + "Valores: " + ", ".join(f"`{v}`" for v in prop["enum"])
        if "default" in prop:
            desc += f" Default: `{prop['default']}`"
        if "example" in prop:
            ex = json.dumps(prop["example"], ensure_ascii=False)
            if len(ex) > 150:
                ex = ex[:147] + "..."
            desc += f" Ex.: `{ex}`"
        rows.append((prefix + name, ptype, "✅" if name in required else "", desc.strip()))
        # Recursão em objetos / arrays de objetos / refs / combiners
        child = prop
        if prop.get("type") == "array":
            child = prop.get("items", {})
        if isinstance(child, dict) and any(k in child for k in ("properties", "$ref", "anyOf", "oneOf")):
            rows += schema_rows(child, prefix + name + ".", ancestors, depth + 1)
    return rows


def render_schema(schema: dict) -> str:
    """Renderiza um schema como tabela markdown de campos (com exemplo se houver)."""
    if not isinstance(schema, dict) or not schema:
        return "_Schema não documentado._"
    label = ""
    if "$ref" in schema:
        label = f"**Schema:** `{resolve_ref(schema['$ref'])[0]}`\n\n"
    rows = schema_rows(schema)
    if rows:
        lines = ["| Campo | Tipo | Obrigatório | Descrição |", "|-------|------|-------------|-----------|"]
        for path, ptype, req, desc in rows:
            lines.append(f"| `{path}` | {ptype} | {req} | {desc} |")
        return label + "\n".join(lines)
    example = schema.get("example")
    if example is not None:
        return label + "Exemplo:\n\n```json\n" + json.dumps(example, indent=2, ensure_ascii=False)[:2000] + "\n```"
    ptype = schema.get("type")
    return label + (f"_Tipo primitivo: `{ptype}`._" if ptype else "_Schema sem propriedades documentadas._")


def render_parameters(op: dict) -> str:
    """Tabela de parâmetros (path/query) do endpoint, com $refs resolvidos."""
    rows = []
    for p in op.get("parameters") or []:
        if not p:
            continue
        if "$ref" in p:
            _, p = resolve_ref(p["$ref"])
        sch = p.get("schema") or {}
        ptype = schema_type_str(sch)
        desc = clean_html(p.get("description", "")).replace("\n", " ").replace("|", "\\|")
        if sch.get("enum"):
            desc += " Valores: " + ", ".join(f"`{v}`" for v in sch["enum"])
        if "default" in sch:
            desc += f" Default: `{sch['default']}`"
        ex = p.get("example", sch.get("example"))
        if ex is not None:
            ex_str = json.dumps(ex, ensure_ascii=False) if not isinstance(ex, str) else ex
            desc += f" Ex.: `{ex_str}`"
        rows.append(
            f"| `{p.get('name', '')}` | {p.get('in', '')} | {'✅' if p.get('required') else ''} | {ptype} | {desc.strip()} |"
        )
    if not rows:
        return ""
    return "| Nome | Local | Obrigatório | Tipo | Descrição |\n|------|-------|-------------|------|-----------|\n" + "\n".join(rows)


def render_request_body(op: dict) -> str:
    """Renderiza o request body: descrição, content-types e schema completo do payload."""
    rb = op.get("requestBody")
    if not rb:
        return ""
    parts = []
    desc = clean_html(rb.get("description", ""))
    if desc:
        parts.append(desc)
        parts.append("")
    if rb.get("required"):
        parts.append("**Body obrigatório.**")
        parts.append("")
    for ctype, media in (rb.get("content") or {}).items():
        parts.append(f"**Content-Type:** `{ctype}`")
        parts.append("")
        parts.append(render_schema((media or {}).get("schema") or {}))
        example = (media or {}).get("example")
        if example is not None:
            parts.append("")
            parts.append("Exemplo:\n\n```json\n" + json.dumps(example, indent=2, ensure_ascii=False)[:3000] + "\n```")
        parts.append("")
    if not rb.get("content"):
        parts.append("_Sem schema de body documentado._")
    return "\n".join(parts).strip()


def render_responses(op: dict) -> str:
    """Renderiza as respostas por status code; cada uma com schema vira um <details> fechado."""
    responses = op.get("responses") or {}
    if not responses:
        return ""
    parts = []
    for code, resp in responses.items():
        if "$ref" in resp:
            _, resp = resolve_ref(resp["$ref"])
        desc = clean_html(resp.get("description", ""))
        summary = f"`{code}` — {desc}"
        inner = []
        for ctype, media in (resp.get("content") or {}).items():
            inner.append(f"**Content-Type:** `{ctype}`")
            inner.append("")
            inner.append(render_schema((media or {}).get("schema") or {}))
            inner.append("")
        if inner:
            parts.append("<details>")
            parts.append(f"<summary>{summary}</summary>")
            parts.append("")
            parts.extend(inner)
            parts.append("</details>")
        else:
            parts.append(f"- {summary}")
        parts.append("")
    return "\n".join(parts).strip()

## Construção do documento

### Parte 1 — Seções conceituais (introdução da API)
Cada seção `##` da descrição do spec vira uma seção da Parte 1 (auth, regiões, paginação, erros, versionamento etc.).

In [27]:
# Seções conceituais da introdução (auth, paginação, erros, versionamento...)
conceptual_sections = []
intro = clean_html(spec["info"].get("description", ""))
for title, body in split_h2_sections(intro):
    conceptual_sections.append((title, f"## {title}\n\n{body}"))

print(f"Seções conceituais: {len(conceptual_sections)}")
for title, _ in conceptual_sections:
    print(" -", title)

Seções conceituais: 5
 - Introduction
 - Getting started
 - Making requests
 - Errors
 - Versioning


### Parte 2 — Endpoints por domínio, com seções colapsáveis
Para cada tag do spec, agregamos todos os endpoints. Cada endpoint recebe uma âncora própria e seções `<details>` **fechadas** para parâmetros, request body (payload completo) e respostas. Cada domínio vira um chunk individual.

In [28]:
def render_endpoint(path: str, method: str, op: dict) -> tuple[str, str]:
    """Renderiza um endpoint completo com âncora e <details> fechados. Retorna (anchor, markdown)."""
    anchor = slugify(f"{method.lower()}-{path}-{op['operationId']}")
    lines = [f'<a id="{anchor}"></a>', "", f"### `{method} {path}` — {op.get('summary', '')}", ""]
    lines.append(f"- **operationId:** `{op['operationId']}`")
    lines.append(f"- **Domínio(s):** {', '.join(op.get('tags', []))}")
    desc = clean_html(op.get("description", ""))
    if desc and desc != op.get("summary", ""):
        lines += ["", desc]

    params_md = render_parameters(op)
    if params_md:
        n_params = len(op.get("parameters") or [])
        lines += ["", "<details>", f"<summary><strong>Parâmetros</strong> ({n_params})</summary>", "", params_md, "", "</details>"]

    rb_md = render_request_body(op)
    if rb_md:
        lines += ["", "<details>", "<summary><strong>Request body (payload)</strong></summary>", "", rb_md, "", "</details>"]

    resp_md = render_responses(op)
    if resp_md:
        n_resp = len(op.get("responses") or {})
        lines += ["", "<details>", f"<summary><strong>Respostas</strong> ({n_resp})</summary>", "", resp_md, "", "</details>"]

    return anchor, "\n".join(lines)


# Monta um chunk por domínio + entradas do índice
domain_chunks = []   # {"id", "title", "tags", "content"}
index_entries = []   # (tag_display, [(anchor, label)])

for tag in spec["tags"]:
    tag_name = tag["name"]
    endpoints = [(p, m, op) for p, m, op in iter_operations() if tag_name in op.get("tags", [])]
    if not endpoints:
        continue
    display = tag.get("x-displayName", tag_name)
    lines = [f"## {display}", ""]
    tag_desc = clean_html(tag.get("description", ""))
    if tag_desc:
        lines += [tag_desc, ""]
    entries = []
    for path, method, op in endpoints:
        anchor, block = render_endpoint(path, method, op)
        entries.append((anchor, f"`{method} {path}` — {op.get('summary', '')}"))
        lines.append(block)
        lines.append("")
    domain_chunks.append({
        "id": f"dominio-{slugify(tag_name)}",
        "title": f"Domínio: {display}",
        "tags": ["dominio", slugify(tag_name)],
        "content": "\n".join(lines).strip(),
    })
    index_entries.append((display, entries))

print(f"Chunks de domínio: {len(domain_chunks)}")
print(f"Total de endpoints documentados: {sum(len(e) for _, e in index_entries)}")

Chunks de domínio: 62
Total de endpoints documentados: 369


## Escrita dos arquivos
1. **Chunks por domínio** em `raw_info/chunks/` — um `.md` por tag, com frontmatter YAML (sem importância — definida via embeddings no RAG) + `index.md`.
2. **Documento único** `raw_info/highbond_api_completo.md` — índice com links, Parte 1 (conceitos) e Parte 2 (referência completa).

In [29]:
# --- 1. Chunks por domínio em raw_info/chunks/ -----------------------------------
# Limpa chunks antigos para manter o diretório consistente
for old in CHUNKS_DIR.glob("*.md"):
    old.unlink()

chunk_index_lines = [
    "# Índice de Chunks — HighBond API (RAG)",
    "",
    "| Chunk | Tamanho (chars) |",
    "|-------|-----------------|",
]

for c in domain_chunks:
    frontmatter = (
        "---\n"
        f"id: {c['id']}\n"
        f"title: \"{c['title']}\"\n"
        f"tags: [{', '.join(c['tags'])}]\n"
        f"source: openapi.json (HighBond API v{spec['info']['version']})\n"
        "---\n\n"
    )
    # No chunk individual, o título vira H1
    body = re.sub(r"^## ", "# ", c["content"], count=1)
    (CHUNKS_DIR / f"{c['id']}.md").write_text(frontmatter + body + "\n", encoding="utf-8")
    chunk_index_lines.append(f"| [{c['id']}]({c['id']}.md) | {len(c['content'])} |")

(CHUNKS_DIR / "index.md").write_text("\n".join(chunk_index_lines) + "\n", encoding="utf-8")

# --- 2. Documento único -------------------------------------------------------
doc = []
doc.append(f"# {spec['info']['title']} — Documentação Completa")
doc.append("")
doc.append(f"Versão da API: **{spec['info']['version']}** — gerado a partir de `openapi.json`")
doc.append("")
doc.append("> Todas as seções de parâmetros, payloads e respostas usam blocos `<details>` fechados por padrão.")
doc.append("")
doc.append("## Índice")
doc.append("")
doc.append("### Conceitos fundamentais")
doc.append("")
for title, _ in conceptual_sections:
    doc.append(f"- [{title}](#{slugify(title)})")
doc.append("")
doc.append("### Endpoints por domínio")
doc.append("")
for display, entries in index_entries:
    doc.append(f"**{display}**")
    doc.append("")
    for anchor, label in entries:
        doc.append(f"- [{label}](#{anchor})")
    doc.append("")

doc.append("---")
doc.append("")
doc.append("# Parte 1 — Conceitos fundamentais")
doc.append("")
for _, section in conceptual_sections:
    doc.append(section)
    doc.append("")
doc.append("---")
doc.append("")
doc.append("# Parte 2 — Referência completa de endpoints")
doc.append("")
for c in domain_chunks:
    doc.append(c["content"])
    doc.append("")

OUT_FILE.write_text("\n".join(doc) + "\n", encoding="utf-8")

size_kb = OUT_FILE.stat().st_size / 1024
print(f"✅ {len(domain_chunks)} chunks + index.md em {CHUNKS_DIR.resolve()}")
print(f"✅ Documento único em {OUT_FILE.resolve()} ({size_kb:.1f} KB)")

✅ 62 chunks + index.md em C:\Users\Rafael Luckner\Documents\repos\uniso\RAG - Contexto para LLMS\raw_info\chunks
✅ Documento único em C:\Users\Rafael Luckner\Documents\repos\uniso\RAG - Contexto para LLMS\raw_info\highbond_api_completo.md (2604.9 KB)


## Validação
Estatísticas do documento gerado e prévia de um endpoint com payload, para conferir a renderização das tabelas e dos blocos colapsáveis.

In [30]:
content = OUT_FILE.read_text(encoding="utf-8")
chunk_sizes = [len(c["content"]) for c in domain_chunks]
print(f"Documento único: {len(content)/1024:.1f} KB")
print(f"Chunks de domínio: {len(chunk_sizes)} | menor: {min(chunk_sizes)} | maior: {max(chunk_sizes)} | média: {sum(chunk_sizes)//len(chunk_sizes)} chars")
print(f"Blocos colapsáveis (<details>): {content.count('<details>')}")
print(f"Âncoras de endpoints: {content.count('<a id=')}")
print(f"Tabelas de payload/schema: {content.count('| Campo | Tipo |')}")
print(f"Tabelas de parâmetros: {content.count('| Nome | Local |')}")

print("\n--- Prévia: primeiro endpoint com request body ---\n")
match = re.search(
    r'<a id="[^"]+"></a>\n\n### `POST[\s\S]+?(?=\n<a id=|\Z)', content
)
if match:
    print(match.group(0)[:3000])
else:
    print(content[:3000])

Documento único: 2539.3 KB
Chunks de domínio: 62 | menor: 2000 | maior: 227760 | média: 40879 chars
Blocos colapsáveis (<details>): 1936
Âncoras de endpoints: 369
Tabelas de payload/schema: 1196
Tabelas de parâmetros: 369

--- Prévia: primeiro endpoint com request body ---

<a id="post-orgs-org-id-robots-createrobot"></a>

### `POST /orgs/{org_id}/robots` — Create a robot

- **operationId:** `createRobot`
- **Domínio(s):** Robots

Create a robot to automate repetitive tasks. The robot contains analytic scripts and related files, and carries out any scheduled tasks that you configure.

<details>
<summary><strong>Parâmetros</strong> (1)</summary>

| Nome | Local | Obrigatório | Tipo | Descrição |
|------|-------|-------------|------|-----------|
| `org_id` | path | ✅ | string | The ID of the HighBond instance. Ex.: `1` |

</details>

<details>
<summary><strong>Request body (payload)</strong></summary>

The data required to create a robot.

**Body obrigatório.**

**Content-Type:** `applic